# L04 · MDPs and the Bellman Equation

## Goal

- define state, action, transition, and reward
- compute a Bellman backup
- interpret value iteration

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L04:toy:42").hexdigest()
print(f"lesson=L04 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L04 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:42f1ff9de0bc274e282f19496e2b5a797a2a3c9f9d59d57b348abed6cb1058d0 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: bandits → **MDPs and Bellman equations** → MC/TD/Q-learning

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)\left[r+\gamma V_k(s')\right]$$

An MDP assumes the current state and action contain everything needed for the future. A Bellman backup decomposes long-run return into one-step reward plus next-state value. Terminal states have no future, so their bootstrap value is zero.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Will a state next to the goal have value above the terminal state's zero value? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>Yes, because reward is received on entering the goal. The terminal state itself does not pay repeatedly.</details>

In [2]:
from rl_study.algorithms.tabular import gridworld_model, value_iteration
from rl_study.envs import TinyGridWorld
grid = TinyGridWorld()
transitions, rewards, terminal = gridworld_model(grid)
dp_result = value_iteration(transitions, rewards, terminal, gamma=0.99)
print("values")
print(dp_result.values.reshape(4, 4).round(decimals=3))
print({"iterations": dp_result.iterations,
       "converged": dp_result.converged,
       "terminal_value": float(dp_result.values[-1])})

values
tensor([[0.9020, 0.9210, 0.9410, 0.9600],
        [0.9210, 0.9410, 0.9600, 0.9800],
        [0.9410, 0.9600, 0.9800, 1.0000],
        [0.9600, 0.9800, 1.0000, 0.0000]])
{'iterations': 7, 'converged': True, 'terminal_value': 0.0}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** With a known transition model, value iteration provides an exact small-grid baseline. It isolates environment and gamma errors before comparison with model-free learning.

**Common trap:** Bootstrapping from a terminal state creates a different MDP with repeatedly collected goal reward. The terminal-zero check fixes the boundary. Regression tests: `test_value_iteration_terminal_value_is_zero`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert dp_result.converged and dp_result.values[-1].item() == 0.0
print("checks=passed")

checks=passed


**Recall:** What does the presence or absence of `max` mean in Bellman expectation versus optimality equations? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** Value iteration converged in 7 iterations and the terminal value is exactly zero. Nonterminal states nearer the goal have larger discounted values.
- Executable checks: `test_value_iteration_terminal_value_is_zero`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L05 estimates these Bellman targets from trajectories without a transition model.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`